
# 00_Determining_Evacuees_and_Evacuation_Motifs

This notebook prepares the evacuation-classification and evacuation-motif inputs used in the Marshall Fire analysis.

It follows the paper's nighttime-location logic and then applies the downstream motif workflow in one place:

1. Load raw stop-level mobility data.
2. Compute nighttime dwell using the paper definition of **8 PM to 7 AM**.
3. Aggregate nighttime dwell to **user × week × CBG**.
4. Create threshold-specific weekly nighttime files for all robustness thresholds.
5. Determine each user's pre-disaster home CBG from the weekly nighttime panels.
6. Identify evacuees and non-evacuees during the disaster window.
7. Build user-level evacuation motifs from the sequence of nighttime locations across post-disaster periods.

In [ ]:

import os
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("mode.chained_assignment", None)


In [ ]:

DATA_DIR = Path(".")
RAW_STOP_DIR = DATA_DIR / "raw_stops"
OUTPUT_DIR = DATA_DIR


THRESHOLDS = [240, 480, 720, 960, 1200, 1440, 1680, 1920]
MAIN_THRESHOLD = 480


# ------------------------------------------------------------------
# Marshall Fire study windows used in this notebook.
# Adjust if your raw-stop files are organized differently.
# ------------------------------------------------------------------
PRE_START = pd.Timestamp("2021-11-08")
PRE_END   = pd.Timestamp("2021-12-05")   # end-exclusive weekly boundary
DISASTER_START = pd.Timestamp("2021-12-30")
DISASTER_END   = pd.Timestamp("2022-01-03")  # covers Dec 30-Jan 2 nights
POST_END = pd.Timestamp("2022-06-06")

# Time zone and nighttime definition from the paper:
# nighttime = 8 PM to 7 AM local time.
LOCAL_TIMEZONE = "US/Mountain"
NIGHT_START_HOUR = 20
NIGHT_END_HOUR = 7

# ------------------------------------------------------------------
# Raw stop columns expected in the private data.
# Rename here if your files use different names.
# ------------------------------------------------------------------
USER_COL = "cuebiq_id"
CBG_COL = "block_group_id"
START_COL = "stop_zoned_datetime"
DWELL_COL = "dwell_time_minutes"

# If your raw-stop files already have an end timestamp, set USE_END_COL = True.
# Otherwise the notebook will infer end time as start time + dwell minutes.
USE_END_COL = False
END_COL = "stop_end_zoned_datetime"


In [ ]:

raw_files = sorted(RAW_STOP_DIR.glob("*.csv")) + sorted(RAW_STOP_DIR.glob("*.csv.gz"))
print("Number of raw stop files found:", len(raw_files))
raw_files[:5]


In [ ]:

all_stops = []

for file_path in raw_files:
    df = pd.read_csv(file_path)

    # Keep only the columns required for nighttime-location construction.
    keep_cols = [USER_COL, CBG_COL, START_COL, DWELL_COL]
    if USE_END_COL and END_COL in df.columns:
        keep_cols.append(END_COL)

    keep_cols = [c for c in keep_cols if c in df.columns]
    df = df[keep_cols].copy()

    all_stops.append(df)

stops = pd.concat(all_stops, ignore_index=True)

print("Raw stop rows:", len(stops))
stops.head()


In [ ]:

stops[USER_COL] = stops[USER_COL].astype(str)
stops[CBG_COL] = stops[CBG_COL].astype(str)

# Parse start timestamps.
stops[START_COL] = pd.to_datetime(stops[START_COL], errors="coerce", utc=True)

# Remove invalid rows.
stops = stops.loc[
    stops[USER_COL].notna()
    & stops[CBG_COL].notna()
    & stops[START_COL].notna()
    & stops[DWELL_COL].notna()
].copy()

# Force dwell to numeric minutes.
stops[DWELL_COL] = pd.to_numeric(stops[DWELL_COL], errors="coerce")
stops = stops.loc[stops[DWELL_COL].notna() & (stops[DWELL_COL] > 0)].copy()

# Build end time.
if USE_END_COL and END_COL in stops.columns:
    stops[END_COL] = pd.to_datetime(stops[END_COL], errors="coerce", utc=True)
else:
    stops[END_COL] = stops[START_COL] + pd.to_timedelta(stops[DWELL_COL], unit="m")

# Convert to local time used in the study region.
stops["start_local"] = stops[START_COL].dt.tz_convert(LOCAL_TIMEZONE)
stops["end_local"] = stops[END_COL].dt.tz_convert(LOCAL_TIMEZONE)

# Keep only rows with positive duration after cleaning.
stops = stops.loc[stops["end_local"] > stops["start_local"]].copy()

print("Clean stop rows:", len(stops))
stops[[USER_COL, CBG_COL, "start_local", "end_local", DWELL_COL]].head()


In [ ]:

# --------------------------------------------------------------
# A stop may overlap one or two nighttime windows depending on
# when it starts and ends. To keep the code simple and explicit,
# we split each stop across candidate local dates and compute
# overlap with the corresponding nighttime window.
# --------------------------------------------------------------
night_parts = []

for row in stops[[USER_COL, CBG_COL, "start_local", "end_local"]].itertuples(index=False):
    uid = row[0]
    cbg = row[1]
    start_local = row[2]
    end_local = row[3]

    start_date = start_local.date()
    end_date = end_local.date()

    # Candidate anchor dates for nighttime windows.
    # The nighttime window anchored on date d is:
    # d 20:00  ->  d+1 07:00
    candidate_dates = pd.date_range(
        pd.Timestamp(start_date) - pd.Timedelta(days=1),
        pd.Timestamp(end_date),
        freq="D"
    )

    for d in candidate_dates:
        night_start = pd.Timestamp(d).tz_localize(LOCAL_TIMEZONE) + pd.Timedelta(hours=NIGHT_START_HOUR)
        night_end = pd.Timestamp(d + pd.Timedelta(days=1)).tz_localize(LOCAL_TIMEZONE) + pd.Timedelta(hours=NIGHT_END_HOUR)

        overlap_start = max(start_local, night_start)
        overlap_end = min(end_local, night_end)

        if overlap_end > overlap_start:
            overlap_minutes = (overlap_end - overlap_start).total_seconds() / 60.0

            night_parts.append(
                {
                    USER_COL: uid,
                    CBG_COL: cbg,
                    "night_anchor_date": pd.Timestamp(d).date(),
                    "night_start_local": night_start,
                    "night_end_local": night_end,
                    "overlap_minutes": overlap_minutes,
                }
            )

night_df = pd.DataFrame(night_parts)

print("Nighttime-overlap rows:", len(night_df))
night_df.head()


In [ ]:

# Sum nighttime minutes for each user and CBG within each anchored night.
night_daily = (
    night_df.groupby([USER_COL, CBG_COL, "night_anchor_date"], as_index=False)["overlap_minutes"]
    .sum()
    .rename(columns={"overlap_minutes": "night_minutes"})
)

night_daily["night_anchor_date"] = pd.to_datetime(night_daily["night_anchor_date"])
night_daily["week_start"] = night_daily["night_anchor_date"] - pd.to_timedelta(
    night_daily["night_anchor_date"].dt.weekday, unit="D"
)

night_weekly = (
    night_daily.groupby([USER_COL, "week_start", CBG_COL], as_index=False)["night_minutes"]
    .sum()
)

print("Weekly nighttime rows:", len(night_weekly))
night_weekly.head()


In [ ]:

# For each threshold, keep only user-week-CBG combinations where the
# nighttime dwell time reaches the threshold.
# Then, within each user-week, rank candidate CBGs by nighttime dwell.
# The top-ranked CBG becomes the weekly nighttime location.

created_threshold_files = {}

for threshold in THRESHOLDS:
    weekly_t = night_weekly.loc[night_weekly["night_minutes"] >= threshold].copy()

    weekly_t = weekly_t.sort_values(
        by=[USER_COL, "week_start", "night_minutes", CBG_COL],
        ascending=[True, True, False, True]
    ).copy()

    weekly_t["rank_in_week"] = weekly_t.groupby([USER_COL, "week_start"]).cumcount() + 1

    weekly_top = weekly_t.loc[weekly_t["rank_in_week"] == 1].copy()

    weekly_top = weekly_top.rename(
        columns={
            CBG_COL: "weekly_night_cbg",
            "night_minutes": "weekly_night_minutes"
        }
    )

    weekly_top["threshold_minutes"] = threshold
    weekly_top = weekly_top[[USER_COL, "week_start", "weekly_night_cbg", "weekly_night_minutes", "threshold_minutes"]].copy()

    out_name = THRESHOLD_OUTPUT_NAMES[threshold]
    out_path = OUTPUT_DIR / out_name
    weekly_top.to_csv(out_path, index=False, compression="gzip")

    created_threshold_files[threshold] = out_path

    print(f"Saved threshold {threshold}: {out_path} | rows = {len(weekly_top)}")


In [ ]:

# ------------------------------------------------------------------
# Now the threshold files exist. From this point onward, the notebook
# follows the same downstream style as the original motif workflow.
# ------------------------------------------------------------------
THRESHOLD_FILES = {
    240: "merged_weekly_no_id_240.csv.gz",
    480: "merged_weekly_no_id.csv.gz",
    720: "merged_weekly_no_id_720.csv.gz",
    960: "merged_weekly_no_id_960.csv.gz",
    1200: "merged_weekly_no_id_1200.csv.gz",
    1440: "merged_weekly_no_id_1440.csv.gz",
    1680: "merged_weekly_no_id_1680.csv.gz",
    1920: "merged_weekly_no_id_1920.csv.gz",
}

threshold_panels = {}

for threshold, filename in THRESHOLD_FILES.items():
    panel = pd.read_csv(OUTPUT_DIR / filename, parse_dates=["week_start"])
    panel[USER_COL] = panel[USER_COL].astype(str)
    panel["weekly_night_cbg"] = panel["weekly_night_cbg"].astype(str)

    threshold_panels[threshold] = panel

    print(threshold, panel.shape)


In [ ]:

threshold_summary = []

for threshold, panel in threshold_panels.items():
    pre_panel = panel.loc[(panel["week_start"] >= PRE_START) & (panel["week_start"] < PRE_END)].copy()

    pre_home = (
        pre_panel.groupby([USER_COL, "weekly_night_cbg"], as_index=False)["weekly_night_minutes"]
        .sum()
        .sort_values([USER_COL, "weekly_night_minutes", "weekly_night_cbg"], ascending=[True, False, True])
        .drop_duplicates(USER_COL)
        .rename(columns={"weekly_night_cbg": "pre_home_cbg", "weekly_night_minutes": "pre_home_total_night_minutes"})
    )

    disaster_panel = panel.loc[(panel["week_start"] >= pd.Timestamp("2021-12-27")) & (panel["week_start"] < pd.Timestamp("2022-01-03"))].copy()
    disaster_panel = disaster_panel.sort_values([USER_COL, "weekly_night_minutes", "weekly_night_cbg"], ascending=[True, False, True])
    disaster_panel = disaster_panel.drop_duplicates(USER_COL)
    disaster_panel = disaster_panel.rename(columns={"weekly_night_cbg": "disaster_week_cbg", "weekly_night_minutes": "disaster_week_night_minutes"})

    evac_df = pre_home.merge(disaster_panel[[USER_COL, "disaster_week_cbg", "disaster_week_night_minutes"]], on=USER_COL, how="left")

    evac_df["evacuated"] = np.where(
        evac_df["disaster_week_cbg"].notna() & (evac_df["disaster_week_cbg"] != evac_df["pre_home_cbg"]),
        1,
        0
    )

    threshold_summary.append(
        {
            "threshold_minutes": threshold,
            "n_users_with_pre_home": evac_df[USER_COL].nunique(),
            "n_users_with_disaster_week_location": int(evac_df["disaster_week_cbg"].notna().sum()),
            "n_evacuated": int(evac_df["evacuated"].sum()),
            "evacuation_rate": float(evac_df["evacuated"].mean()) if len(evac_df) > 0 else np.nan,
        }
    )

threshold_summary = pd.DataFrame(threshold_summary).sort_values("threshold_minutes").reset_index(drop=True)
threshold_summary


In [ ]:

weekly_main = threshold_panels[MAIN_THRESHOLD].copy()

# Keep only the main analysis horizon.
weekly_main = weekly_main.loc[
    (weekly_main["week_start"] >= PRE_START) & (weekly_main["week_start"] <= POST_END)
].copy()

print(weekly_main.shape)
weekly_main.head()


In [ ]:

# ----------------------------
# Pre-disaster home CBG
# ----------------------------
pre_main = weekly_main.loc[(weekly_main["week_start"] >= PRE_START) & (weekly_main["week_start"] < PRE_END)].copy()

pre_home_main = (
    pre_main.groupby([USER_COL, "weekly_night_cbg"], as_index=False)["weekly_night_minutes"]
    .sum()
    .sort_values([USER_COL, "weekly_night_minutes", "weekly_night_cbg"], ascending=[True, False, True])
    .drop_duplicates(USER_COL)
    .rename(columns={"weekly_night_cbg": "pre_crisis_home_cbg", "weekly_night_minutes": "pre_home_total_night_minutes"})
)

# ----------------------------
# Disaster-week nighttime CBG
# ----------------------------
crisis_main = weekly_main.loc[
    (weekly_main["week_start"] >= pd.Timestamp("2021-12-27")) &
    (weekly_main["week_start"] < pd.Timestamp("2022-01-03"))
].copy()

crisis_main = (
    crisis_main.sort_values([USER_COL, "weekly_night_minutes", "weekly_night_cbg"], ascending=[True, False, True])
    .drop_duplicates(USER_COL)
    .rename(columns={"weekly_night_cbg": "crisis_home_cbg", "weekly_night_minutes": "crisis_week_night_minutes"})
)

evacuation_status = pre_home_main.merge(
    crisis_main[[USER_COL, "crisis_home_cbg", "crisis_week_night_minutes"]],
    on=USER_COL,
    how="left"
)

evacuation_status["evacuation_status"] = np.where(
    evacuation_status["crisis_home_cbg"].notna() &
    (evacuation_status["crisis_home_cbg"] != evacuation_status["pre_crisis_home_cbg"]),
    "evacuated",
    "not_evacuated"
)

evacuation_status.head()


In [ ]:

# Create a wide user-week panel for the post-disaster periods used in motifs.
post_main = weekly_main.loc[weekly_main["week_start"] >= pd.Timestamp("2022-01-03")].copy()

# Rank within each user-week and keep the top nightly CBG.
post_main = post_main.sort_values([USER_COL, "week_start", "weekly_night_minutes", "weekly_night_cbg"], ascending=[True, True, False, True]).copy()
post_main["rank_in_week"] = post_main.groupby([USER_COL, "week_start"]).cumcount() + 1
post_main = post_main.loc[post_main["rank_in_week"] == 1].copy()

# Representative post-disaster periods.
# You can adjust these windows if your final paper version used slightly different week groupings.
immediate_weeks = pd.date_range("2022-01-03", "2022-01-16", freq="W-MON")
short_term_weeks = pd.date_range("2022-01-17", "2022-02-13", freq="W-MON")
long_term_weeks = pd.date_range("2022-02-14", "2022-06-06", freq="W-MON")

immediate = post_main.loc[post_main["week_start"].isin(immediate_weeks)].copy()
short_term = post_main.loc[post_main["week_start"].isin(short_term_weeks)].copy()
long_term = post_main.loc[post_main["week_start"].isin(long_term_weeks)].copy()

immediate_loc = (
    immediate.groupby([USER_COL, "weekly_night_cbg"], as_index=False)["weekly_night_minutes"]
    .sum()
    .sort_values([USER_COL, "weekly_night_minutes", "weekly_night_cbg"], ascending=[True, False, True])
    .drop_duplicates(USER_COL)
    .rename(columns={"weekly_night_cbg": "immediately_post_crisis_CBG"})
)

short_term_loc = (
    short_term.groupby([USER_COL, "weekly_night_cbg"], as_index=False)["weekly_night_minutes"]
    .sum()
    .sort_values([USER_COL, "weekly_night_minutes", "weekly_night_cbg"], ascending=[True, False, True])
    .drop_duplicates(USER_COL)
    .rename(columns={"weekly_night_cbg": "short_term_displacement_CBG"})
)

long_term_loc = (
    long_term.groupby([USER_COL, "weekly_night_cbg"], as_index=False)["weekly_night_minutes"]
    .sum()
    .sort_values([USER_COL, "weekly_night_minutes", "weekly_night_cbg"], ascending=[True, False, True])
    .drop_duplicates(USER_COL)
    .rename(columns={"weekly_night_cbg": "long_term_displacement_CBG"})
)

merged_weekly = evacuation_status[[USER_COL, "evacuation_status", "pre_crisis_home_cbg", "crisis_home_cbg"]].merge(
    immediate_loc[[USER_COL, "immediately_post_crisis_CBG"]],
    on=USER_COL, how="left"
).merge(
    short_term_loc[[USER_COL, "short_term_displacement_CBG"]],
    on=USER_COL, how="left"
).merge(
    long_term_loc[[USER_COL, "long_term_displacement_CBG"]],
    on=USER_COL, how="left"
)

merged_weekly.head()


In [ ]:

# Replace string-like missing values consistently.
for col in [
    "pre_crisis_home_cbg",
    "crisis_home_cbg",
    "immediately_post_crisis_CBG",
    "short_term_displacement_CBG",
    "long_term_displacement_CBG",
]:
    merged_weekly[col] = merged_weekly[col].replace({"nan": np.nan, "None": np.nan, "": np.nan})

# Collapse short-term and long-term into one downstream location when needed.
merged_weekly["long_term"] = merged_weekly["short_term_displacement_CBG"].combine_first(
    merged_weekly["long_term_displacement_CBG"]
)

# Count distinct observed locations across the motif stages.
stage_cols = [
    "pre_crisis_home_cbg",
    "crisis_home_cbg",
    "immediately_post_crisis_CBG",
    "long_term",
]

unique_counts = []
for row in merged_weekly[stage_cols].itertuples(index=False):
    vals = [x for x in row if pd.notna(x)]
    unique_counts.append(len(set(vals)))

merged_weekly["unique_cbgs"] = unique_counts

# User type logic kept simple and close to the original motif notebook.
merged_weekly["user_type"] = np.nan
merged_weekly.loc[merged_weekly["unique_cbgs"] == 1, "user_type"] = 1
merged_weekly.loc[merged_weekly["unique_cbgs"] == 2, "user_type"] = 2
merged_weekly.loc[merged_weekly["unique_cbgs"] >= 3, "user_type"] = 3

# Force never-returned evacuees into a distinct type.
merged_weekly.loc[
    (merged_weekly["evacuation_status"] == "evacuated") &
    merged_weekly["long_term"].notna() &
    (merged_weekly["long_term"] != merged_weekly["pre_crisis_home_cbg"]),
    "user_type"
] = 4

merged_weekly.head()


In [ ]:

motifs = []

for row in merged_weekly.itertuples(index=False):
    uid = getattr(row, USER_COL)
    user_type = row.user_type
    pre_home = row.pre_crisis_home_cbg
    crisis_home = row.crisis_home_cbg
    immediate_post = row.immediately_post_crisis_CBG
    short_term_loc = row.short_term_displacement_CBG
    long_term_loc = row.long_term

    motif = np.nan

    # --------------------------------------------------------------
    # Type 1: Stable in all observed periods.
    # --------------------------------------------------------------
    if user_type == 1:
        motif = "1"

    # --------------------------------------------------------------
    # Type 2: Two distinct observed locations across stages.
    # --------------------------------------------------------------
    elif user_type == 2:
        if pd.notna(crisis_home) and (crisis_home == pre_home) and pd.notna(immediate_post) and (immediate_post != pre_home):
            motif = "2-1"   # not evacuated in crisis week, moved later
        elif pd.notna(crisis_home) and (crisis_home != pre_home) and pd.notna(long_term_loc) and (long_term_loc == pre_home):
            motif = "2-2"   # evacuated and returned
        elif pd.notna(crisis_home) and (crisis_home != pre_home) and pd.notna(long_term_loc) and (long_term_loc != pre_home):
            motif = "2-3"   # evacuated and settled elsewhere
        else:
            motif = "2-other"

    # --------------------------------------------------------------
    # Type 3: Three or more locations over time.
    # --------------------------------------------------------------
    elif user_type == 3:
        if pd.notna(crisis_home) and (crisis_home != pre_home) and pd.notna(long_term_loc) and (long_term_loc == pre_home):
            motif = "3-1"   # evacuated with multi-step path, eventually returned
        elif pd.notna(crisis_home) and (crisis_home != pre_home) and pd.notna(long_term_loc) and (long_term_loc != pre_home):
            motif = "3-2"   # evacuated with multi-step path, never returned
        elif pd.notna(crisis_home) and (crisis_home == pre_home) and pd.notna(immediate_post) and (immediate_post != pre_home):
            motif = "3-3"   # delayed movement after crisis week
        else:
            motif = "3-other"

    # --------------------------------------------------------------
    # Type 4: Explicit never-returned evacuee.
    # --------------------------------------------------------------
    elif user_type == 4:
        motif = "7"

    motifs.append(motif)

merged_weekly["motif"] = motifs
merged_weekly.head()


In [ ]:

motif_final = []

for m in merged_weekly["motif"]:
    if m == "1":
        motif_final.append("not_evacuated_atall")
    elif m in ["2-1", "3-3"]:
        motif_final.append("Not_Evacuated_but_migrated")
    elif m in ["2-2", "3-1"]:
        motif_final.append("Evacuated_and_Returned")
    elif m in ["2-3", "3-2", "7"]:
        motif_final.append("Evacuated_never_Returned")
    else:
        motif_final.append("other_or_unclassified")

merged_weekly["motif_final"] = motif_final

merged_weekly["motif_not_evacuated"] = np.where(
    merged_weekly["motif_final"].isin(["not_evacuated_atall", "Not_Evacuated_but_migrated"]),
    1, 0
)

merged_weekly["motif_Evacuated_and_Returned"] = np.where(
    merged_weekly["motif_final"] == "Evacuated_and_Returned",
    1, 0
)

merged_weekly["motif_Evacuated_never_Returned"] = np.where(
    merged_weekly["motif_final"] == "Evacuated_never_Returned",
    1, 0
)

merged_weekly["motif_final"].value_counts(dropna=False)


In [ ]:

threshold_summary.to_csv(OUTPUT_DIR / "threshold_robustness_summary.csv", index=False)

evacuation_status.to_csv(OUTPUT_DIR / "evacuation_status.csv.gz", index=False, compression="gzip")

merged_weekly.to_csv(OUTPUT_DIR / "final_motifs.csv.gz", index=False, compression="gzip")

print("Saved:")
print("-", OUTPUT_DIR / "threshold_robustness_summary.csv")
print("-", OUTPUT_DIR / "evacuation_status.csv.gz")
print("-", OUTPUT_DIR / "final_motifs.csv.gz")
